[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C29_Frontier_Interp_Course/04_steering/04_steering.ipynb)

# 04 · 特征 Steering（从零做激活干预，量化 on/off-target）

目标：在**合成叠加数据** + **玩具行为读出**上，用 numpy 从零实现 **ActAdd / SAE 特征 clamp / difference-of-means** 三种 steering，做**强度扫描**、量化 **on-target 效果与 off-target 副作用**、并用**方向消融**移除行为。

路线：数据+行为读出 → ActAdd 剂量反应 → SAE 特征 clamp → difference-of-means 对比 → 强度窗口(over-steer 崩坏) → off-target 副作用 → 方向消融 → ✏️ 练习 ×4 → 📖 答案 → 🧪 真实 steering 配置胶囊。

> 心智模型：**steering = 在激活空间加一个向量**。难点不在加，而在**加对方向、加准强度、不伤无辜**。

## 1 · 合成数据 + 玩具行为读出

复用模块 01 的叠加数据：`d` 维激活由 `m_true` 个真特征稀疏叠加而成。指定一个**目标特征** `TGT`，它的真方向 `v_tgt` 就是我们要操控的概念。

再定义一个**行为读出** `readout(x, v)`：对激活在某方向上的投影做 logistic，模拟「模型表现出某行为的概率」。目标行为读 `v_tgt`；另准备若干**无关**读出（其它特征方向）用于后面度量副作用。

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

def make_data(n, d=20, m_true=40, p_active=0.04, amp=(0.6, 1.4), noise=0.01, seed=0):
    '''叠加数据：返回 X[n,d], S[n,m_true] 真激活码, F[m_true,d] 真特征方向(单位行)。'''
    r = np.random.default_rng(seed)
    F = r.standard_normal((m_true, d)); F /= np.linalg.norm(F, axis=1, keepdims=True)
    S = (r.random((n, m_true)) < p_active).astype(float)
    S *= r.uniform(*amp, size=(n, m_true))
    X = S @ F + noise * r.standard_normal((n, d))
    return X, S, F

X, S, F = make_data(8000, d=20, m_true=40)
d = 20
TGT = 0                      # 目标特征下标（ground truth）
v_tgt = F[TGT]               # 目标概念的真方向（单位）
OTHERS = [5, 10, 15, 20, 25] # 无关行为：其它特征方向（量副作用用）

def sigmoid(z): return 1.0 / (1.0 + np.exp(-z))
def readout(x, v, scale=4.0, bias=-1.0):
    '''玩具行为读出：激活在方向 v 上的投影 -> logistic 概率。'''
    return sigmoid(scale * (x @ v) + bias)

b0 = readout(X, v_tgt).mean()
print(f'd={d}, 目标特征={TGT}, ||v_tgt||={np.linalg.norm(v_tgt):.3f}')
print(f'baseline 目标行为读出均值 = {b0:.3f}')
assert abs(np.linalg.norm(v_tgt) - 1) < 1e-9
assert 0 < b0 < 0.6, 'baseline 应较低（目标行为多数时候不触发）'
print('✅ 数据与行为读出就绪（目标方向 + 无关方向都已知）')

## 2 · Activation Addition（ActAdd）：加一个方向

最简 steering：`h ← h + α·v_tgt`。扫强度 `α`，看目标行为读出是否**单调上升**——这就是 steering 的剂量-反应曲线最朴素的形态。

In [ ]:
def act_add(x, v, alpha):
    '''ActAdd: 给每个激活加上 alpha * v。'''
    return x + alpha * v

alphas = np.linspace(0, 3, 7)
resp = np.array([readout(act_add(X, v_tgt, a), v_tgt).mean() for a in alphas])
print('α      :', np.round(alphas, 2))
print('目标读出:', np.round(resp, 3))
# 单调不降
assert np.all(np.diff(resp) >= -1e-9), 'ActAdd 应使目标行为单调上升'
assert resp[-1] - resp[0] > 0.3, '强度足够时效果应显著'
print(f'✅ ActAdd 有效：α 从 0→3，目标读出 {resp[0]:.2f}→{resp[-1]:.2f}（单调上升）')

## 3 · SAE 特征 Steering：钳制单义特征

先训一个紧凑 SAE 拿到**单义特征方向**，找到与目标真特征匹配的那个 SAE 特征，然后做 **clamp**：编码 → 把该特征设到目标值 → 解码。

这正是 Golden Gate Claude 的做法（把「金门大桥」特征钳到极高）。我们验证 clamp **确实把特征设到了目标值**，且目标行为随之上升。

In [ ]:
def sae_forward(x, We, be, Wd, bd):
    pre = (x - bd) @ We.T + be
    f = np.maximum(pre, 0.0)
    return f, f @ Wd.T + bd

def train_sae(X, m, lr=0.002, lam=0.3, steps=4000, batch=512, seed=1):
    '''紧凑 L1 SAE（Adam）。返回参数 dict。'''
    r = np.random.default_rng(seed); n, dd = X.shape
    P = dict(We=r.standard_normal((m, dd)) * 0.1, be=np.zeros(m), bd=X.mean(0).copy())
    Wd = r.standard_normal((dd, m)); Wd /= np.linalg.norm(Wd, axis=0, keepdims=True); P['Wd'] = Wd
    mom = {k: np.zeros_like(v) for k, v in P.items()}
    vel = {k: np.zeros_like(v) for k, v in P.items()}
    for t in range(1, steps + 1):
        idx = r.integers(0, n, size=batch); xb = X[idx]
        pre = (xb - P['bd']) @ P['We'].T + P['be']; f = np.maximum(pre, 0.0)
        xhat = f @ P['Wd'].T + P['bd']; resid = xhat - xb
        gr = (2.0 / batch) * resid
        gf = gr @ P['Wd'] + (lam / batch) * (f > 0); gf = gf * (f > 0)
        g = dict(We=gf.T @ (xb - P['bd']), Wd=gr.T @ f, be=gf.sum(0), bd=gr.sum(0))
        for k in P:
            mom[k] = 0.9 * mom[k] + 0.1 * g[k]
            vel[k] = 0.999 * vel[k] + 0.001 * g[k] ** 2
            P[k] -= lr * (mom[k] / (1 - 0.9 ** t)) / (np.sqrt(vel[k] / (1 - 0.999 ** t)) + 1e-8)
        P['Wd'] /= (np.linalg.norm(P['Wd'], axis=0, keepdims=True) + 1e-8)
    return P

sae = train_sae(X, m=80, steps=4000, seed=1)
# 找与目标真特征匹配的 SAE 特征（最大 |cos|），记下符号
D = sae['Wd'] / np.linalg.norm(sae['Wd'], axis=0, keepdims=True)
cos_to_tgt = D.T @ v_tgt
j = int(np.abs(cos_to_tgt).argmax())
sgn = float(np.sign(cos_to_tgt[j]))
print(f'SAE 特征 {j} 匹配目标真特征，|cos|={abs(cos_to_tgt[j]):.3f}，符号={sgn:+.0f}')
assert abs(cos_to_tgt[j]) > 0.85, 'SAE 应学到目标特征'
print('✅ 找到对应目标概念的单义 SAE 特征')

In [ ]:
def clamp_decode(x, sae, j, c):
    '''SAE 特征 clamp：编码 -> 把特征 j 设到 c -> 解码。返回 (h', f_after).'''
    f, _ = sae_forward(x, sae['We'], sae['be'], sae['Wd'], sae['bd'])
    f2 = f.copy(); f2[:, j] = c
    h2 = f2 @ sae['Wd'].T + sae['bd']
    return h2, f2

# clamp 值要沿符号校正方向（解码器列 j ≈ sgn * v_tgt）
clamp_vals = [0, 1, 2, 3, 5]
resp_clamp = np.array([readout(clamp_decode(X, sae, j, sgn * c)[0], v_tgt).mean()
                       for c in clamp_vals])
print('clamp 值 :', clamp_vals)
print('目标读出 :', np.round(resp_clamp, 3))
# 验证 clamp 真的把特征设到了目标值
_, f_after = clamp_decode(X, sae, j, sgn * 3.0)
assert np.allclose(f_after[:, j], sgn * 3.0), 'clamp 应把特征 j 精确设到目标值'
assert resp_clamp[-1] - resp_clamp[0] > 0.3, 'clamp 应显著抬升目标行为'
print('✅ SAE 特征 clamp 有效：钳大特征 -> 目标行为上升（Golden Gate 同款机制）')

## 4 · Difference-of-Means / CAA：强基线

取「目标特征 ON」与「OFF」两组激活的**均值差**作 steering 方向（CAA, Rimsky 2023）。验证它**几乎等于真目标方向**（高余弦），且 steering 效果与真方向一致——这就是为什么 difference-of-means 是难以击败的强基线。

In [ ]:
on = S[:, TGT] > 0     # 目标特征激活的样本
off = ~on
v_md = X[on].mean(0) - X[off].mean(0)          # difference-of-means 方向
v_md_u = v_md / np.linalg.norm(v_md)
cos_md = float(v_md_u @ v_tgt)
print(f'difference-of-means 方向: ||v_md||={np.linalg.norm(v_md):.3f}, cos(与真方向)={cos_md:.3f}')
assert cos_md > 0.9, 'mean-diff 应高度对齐真目标方向'

# 在匹配的【单位】方向上对比 steering 效果：真方向 vs mean-diff
for name, vv in [('真方向 ', v_tgt), ('mean-diff', v_md_u)]:
    r = [readout(act_add(X, vv, a), v_tgt).mean() for a in [0, 1, 2]]
    print(f'{name} α=0,1,2 -> {np.round(r, 3)}')
r_true = readout(act_add(X, v_tgt, 2.0), v_tgt).mean()
r_md = readout(act_add(X, v_md_u, 2.0), v_tgt).mean()
assert abs(r_true - r_md) < 0.1, '两方向的 steering 效果应几乎一致'
print('✅ difference-of-means ≈ 真方向（cos>0.9），效果几乎一致 —— 难击败的强基线')

## 5 · 强度窗口：over-steer 会崩坏

steering 太猛会把激活推出训练分布，模型崩坏。我们用**激活范数**作连贯性代理：随 `α` 增大，目标效果先升到饱和，而激活范数**线性爆炸**——超过自然尺度太多就意味着 over-steer。

In [ ]:
nat_norm = np.linalg.norm(X, axis=1).mean()      # 自然激活范数
alphas = np.array([0, 1, 2, 4, 8, 16], dtype=float)
eff = np.array([readout(act_add(X, v_tgt, a), v_tgt).mean() for a in alphas])
nrm = np.array([np.linalg.norm(act_add(X, v_tgt, a), axis=1).mean() for a in alphas])
print(f'{"α":>4} {"目标效果":>8} {"激活范数":>8} {"范数/自然":>8}')
for a, e, nn in zip(alphas, eff, nrm):
    print(f'{a:>4.0f} {e:>8.3f} {nn:>8.2f} {nn/nat_norm:>8.1f}x')
# 效果会饱和(趋于1)，范数线性爆炸
assert eff[-1] >= eff[2] - 1e-9 and eff[-1] > 0.9, '大强度下效果饱和到高位'
assert nrm[-1] > 3 * nrm[0], 'over-steer: 激活范数远超自然尺度（崩坏信号）'
# 有效窗口：效果达 0.8 的最小 α，但范数还没爆（<3x）
ok = np.where((eff >= 0.8) & (nrm < 3 * nat_norm))[0]
print(f'\n有效窗口内的 α（效果≥0.8 且 范数<3x自然）: {alphas[ok].tolist()}')
print('✅ 看清剂量-反应：效果饱和 + 范数爆炸 = 必须在有效窗口内 steering')

## 6 · off-target 副作用 + 方向消融

**off-target** = steering 对无关行为的副作用。对比**干净方向**（真单义方向）与**混杂方向**（真方向 + 大量噪声）：在**同等 on-target 效果**下，干净方向的副作用应明显更小。

再做**方向消融**（投影消除目标方向），在「目标特征本就激活」的样本上验证它**移除目标行为**、却**保留无关行为**。

In [ ]:
def off_target(x_steered, x_orig, dirs):
    '''无关读出的平均绝对变化（副作用）。'''
    return float(np.mean([abs(readout(x_steered, F[k]).mean() - readout(x_orig, F[k]).mean())
                          for k in dirs]))

def steer_to_level(vdir, level=0.85, amax=8.0):
    '''找使目标读出达到 level 的最小 alpha（匹配 on-target 效果）。'''
    for a in np.linspace(0, amax, 81):
        if readout(act_add(X, vdir, a), v_tgt).mean() >= level:
            return a
    return amax

# 混杂方向：真方向 + 大噪声（模拟未净化的 ActAdd 向量）
rng2 = np.random.default_rng(3)
v_dirty = v_tgt + 0.8 * rng2.standard_normal(d); v_dirty /= np.linalg.norm(v_dirty)
a_clean = steer_to_level(v_tgt); a_dirty = steer_to_level(v_dirty)
ot_clean = off_target(act_add(X, v_tgt, a_clean), X, OTHERS)
ot_dirty = off_target(act_add(X, v_dirty, a_dirty), X, OTHERS)
print(f'干净方向: α={a_clean:.2f}  off-target={ot_clean:.4f}')
print(f'混杂方向: α={a_dirty:.2f}  off-target={ot_dirty:.4f}')
assert ot_clean < ot_dirty, '同等 on-target 下，干净单义方向副作用应更小'
print('✅ 方向越纯，off-target 越低 —— 这正是 SAE 单义特征/净化方向的价值')

In [ ]:
def ablate(x, v):
    '''方向消融：把 x 在 v 上的分量投影清零。'''
    v = v / np.linalg.norm(v)
    return x - (x @ v)[:, None] * v[None, :]

X_ab = ablate(X, v_tgt)
on = S[:, TGT] > 0                       # 在目标特征激活的样本上看效果
r_on_orig = readout(X[on], v_tgt).mean()
r_on_ab = readout(X_ab[on], v_tgt).mean()
ot_ab = off_target(X_ab, X, OTHERS)
print(f'目标行为(激活子集): 原始={r_on_orig:.3f} -> 消融后={r_on_ab:.3f}')
print(f'消融对无关行为的副作用 = {ot_ab:.4f}')
assert r_on_ab < r_on_orig - 0.2, '消融应显著移除目标行为'
assert ot_ab < 0.05, '消融应基本保留无关行为'
print('✅ 方向消融：精准移除目标行为，无关行为几乎不动（refusal 越狱同款机制）')

---
## ✏️ 练习 1：实现 error-preserving 特征 clamp

朴素 clamp 会把 SAE 的**重建误差**也一并丢掉（冒充成副作用）。实现 **error-preserving clamp**：
`h' = Dec(clamp(Enc(h))) + (h − Dec(Enc(h)))`，即只改目标特征、把 SAE 重建残差原样保留。

实现 `clamp_preserve(x, sae, j, c)`，返回 `h'`。验证：当 `c` 等于特征 `j` 的**原始激活值**时，`h'` 应等于原始 `h`（因为什么都没改）。

In [ ]:
def clamp_preserve(x, sae, j, c):
    # TODO:
    #   f, xhat = sae_forward(...)        # xhat = Dec(Enc(x))
    #   resid = x - xhat                  # SAE 重建残差
    #   f2 = f.copy(); f2[:, j] = c       # 钳制目标特征
    #   h2 = f2 @ sae['Wd'].T + sae['bd'] # 重解码
    #   return h2 + resid                 # 保留残差
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
f_orig, _ = sae_forward(X, sae['We'], sae['be'], sae['Wd'], sae['bd'])
# 用每个样本【自己】的特征 j 原值做 clamp -> 应还原 h（恒等）
h_identity = np.vstack([clamp_preserve(X[i:i+1], sae, j, f_orig[i, j]) for i in range(0, 200, 1)])
assert np.allclose(h_identity, X[:200], atol=1e-8), 'clamp 到原值 + 保留残差 应恒等还原'
# 钳大特征仍应抬升目标行为
h_steered = clamp_preserve(X, sae, j, sgn * 4.0)
assert readout(h_steered, v_tgt).mean() > readout(X, v_tgt).mean() + 0.2
print('✅ 练习 1 通过：error-preserving clamp 恒等还原 + 仍能 steering')

## ✏️ 练习 2：找最佳 steering 强度（有效窗口）

实现 `best_alpha(vdir, level=0.85, norm_budget=2.0)`：在 `α∈linspace(0,8,81)` 中，找使目标读出达到 `level`、**且**激活范数不超过 `norm_budget × 自然范数` 的**最小** `α`；找不到返回 `None`。

这就是「效果够强、还没崩坏」的最佳剂量。

In [ ]:
def best_alpha(vdir, level=0.85, norm_budget=2.0):
    nat = np.linalg.norm(X, axis=1).mean()
    # TODO: 遍历 alpha in np.linspace(0,8,81)；
    #   eff = readout(act_add(X, vdir, alpha), v_tgt).mean()
    #   nrm = np.linalg.norm(act_add(X, vdir, alpha), axis=1).mean()
    #   若 eff>=level 且 nrm <= norm_budget*nat: 返回该 alpha
    #   循环结束都没有 -> 返回 None
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
a_star = best_alpha(v_tgt, level=0.85, norm_budget=2.0)
print(f'最佳 steering 强度 α* = {a_star}')
assert a_star is not None and 0 < a_star <= 8
# 在 α* 处效果达标且范数没爆
nat = np.linalg.norm(X, axis=1).mean()
assert readout(act_add(X, v_tgt, a_star), v_tgt).mean() >= 0.85
assert np.linalg.norm(act_add(X, v_tgt, a_star), axis=1).mean() <= 2.0 * nat
# 更小的预算应给出 <= 的可行 α 或 None（更严格）
print('✅ 练习 2 通过：在范数预算内找到达标的最小强度')

## ✏️ 练习 3：off-target 副作用度量

实现 `side_effect(x_steered, x_orig, dirs)` = 无关方向读出的平均绝对变化（同正文 `off_target`）。

用它验证一个核心结论：**方向消融**目标方向，对无关行为的副作用，应**小于**用一个混杂方向做 ActAdd 到同等 on-target 效果的副作用。

In [ ]:
def side_effect(x_steered, x_orig, dirs):
    # TODO: 返回 dirs 中每个方向 readout 均值的 |变化| 的平均
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
se_ab = side_effect(ablate(X, v_tgt), X, OTHERS)               # 消融的副作用
a_dirty = steer_to_level(v_dirty)
se_dirty = side_effect(act_add(X, v_dirty, a_dirty), X, OTHERS) # 混杂方向 ActAdd 的副作用
print(f'消融副作用={se_ab:.4f}   混杂方向ActAdd副作用={se_dirty:.4f}')
assert se_ab >= 0 and se_dirty >= 0
assert se_ab < se_dirty, '精准消融的副作用应小于混杂方向 steering'
print('✅ 练习 3 通过：副作用度量能区分「精准」与「粗暴」干预')

## ✏️ 练习 4：组合干预（同时 steer 两个特征）

实现 `combine_steer(x, dirs_alphas)`：`dirs_alphas` 是 `[(v1, α1), (v2, α2), ...]`，对 `x` 依次加上每个 `αᵢ·vᵢ`，返回结果。

验证：同时 steer 目标特征 `v_tgt` 与另一个特征 `F[7]`，**两个对应行为都上升**。

In [ ]:
def combine_steer(x, dirs_alphas):
    # TODO: h = x.copy(); 对每个 (v, a) in dirs_alphas: h = h + a*v; 返回 h
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
v2 = F[7]
Xc = combine_steer(X, [(v_tgt, 2.0), (v2, 2.0)])
r1_0, r1_1 = readout(X, v_tgt).mean(), readout(Xc, v_tgt).mean()
r2_0, r2_1 = readout(X, v2).mean(), readout(Xc, v2).mean()
print(f'目标特征行为 {r1_0:.3f} -> {r1_1:.3f}')
print(f'第二特征行为 {r2_0:.3f} -> {r2_1:.3f}')
assert r1_1 > r1_0 + 0.2 and r2_1 > r2_0 + 0.2, '组合干预应同时抬升两个行为'
print('✅ 练习 4 通过：可组合多特征干预（如 +诚实 且 −谄媚 的玩具版）')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def clamp_preserve(x, sae, j, c):
    f, xhat = sae_forward(x, sae['We'], sae['be'], sae['Wd'], sae['bd'])
    resid = x - xhat                       # SAE 重建残差，原样保留
    f2 = f.copy(); f2[:, j] = c
    h2 = f2 @ sae['Wd'].T + sae['bd']
    return h2 + resid

In [ ]:
# 练习 2 参考答案
def best_alpha(vdir, level=0.85, norm_budget=2.0):
    nat = np.linalg.norm(X, axis=1).mean()
    for alpha in np.linspace(0, 8, 81):
        eff = readout(act_add(X, vdir, alpha), v_tgt).mean()
        nrm = np.linalg.norm(act_add(X, vdir, alpha), axis=1).mean()
        if eff >= level and nrm <= norm_budget * nat:
            return float(alpha)
    return None

In [ ]:
# 练习 3 参考答案
def side_effect(x_steered, x_orig, dirs):
    return float(np.mean([abs(readout(x_steered, F[k]).mean() - readout(x_orig, F[k]).mean())
                          for k in dirs]))

In [ ]:
# 练习 4 参考答案
def combine_steer(x, dirs_alphas):
    h = x.copy()
    for v, a in dirs_alphas:
        h = h + a * v
    return h

---
## 🧪 真实数据胶囊：真实 steering 的设置与取舍

下面是几个**真实 steering 工作**的公开设定（约数 / 定性）。用它们体会：真实 steering 同样在 on-target 与 off-target、强度窗口之间权衡——你刚在玩具上做的，正是它们的缩影。

In [ ]:
# 真实 steering 工作的公开设定（约数 / 定性）
STEERING = {
    'Golden Gate Claude (Templeton24)': dict(method='SAE feature clamp', target='Golden Gate Bridge 特征',
                                            note='钳到 ~10x 最大激活；过强则语无伦次'),
    'CAA on Llama-2 (Rimsky23)':        dict(method='difference-of-means', target='sycophancy 等行为',
                                            note='单层加 mean-diff 向量；扫系数找窗口'),
    'Refusal dir (Arditi24)':           dict(method='direction ablation/add', target='refusal 行为',
                                            note='消融单方向即越狱；加强则过度拒绝'),
}
for name, s in STEERING.items():
    print(f'- {name}')
    print(f'    方法={s["method"]:<22} 目标={s["target"]}')
    print(f'    {s["note"]}')
print('\n共性：方向(SAE特征/mean-diff) + 强度扫描 + 监控副作用/崩坏 —— 与本 notebook 完全一致。')

**🧪 胶囊练习**：把「on-target vs off-target 权衡」量化成一个分数。实现 `steer_quality(on_target_gain, off_target_cost, eps=1e-6)` = `on_target_gain / (off_target_cost + eps)`（越大越好：效果大、副作用小）。用它判断「干净方向」是否优于「混杂方向」。

In [ ]:
def steer_quality(on_target_gain, off_target_cost, eps=1e-6):
    # TODO: 返回 on_target_gain / (off_target_cost + eps)
    raise NotImplementedError

In [ ]:
# 自测
# 干净方向 vs 混杂方向：同等 on-target gain，比 off-target cost
gain = 0.85 - readout(X, v_tgt).mean()        # 大致的 on-target 提升
q_clean = steer_quality(gain, ot_clean)       # ot_clean 来自第6节
q_dirty = steer_quality(gain, ot_dirty)
print(f'干净方向质量={q_clean:.2f}  混杂方向质量={q_dirty:.2f}')
assert q_clean > q_dirty, '同等效果下，副作用更小者质量更高'
print('✅ 胶囊练习通过：质量分把「精准 steering」量化了出来')

In [ ]:
# 📖 胶囊参考答案
def steer_quality(on_target_gain, off_target_cost, eps=1e-6):
    return on_target_gain / (off_target_cost + eps)

### 小结
- **steering = 激活空间加向量**：ActAdd `h+αv`、SAE 特征 clamp（编码→设值→解码）、方向消融 `h-(v̂ᵀh)v̂`。
- **方向从哪来**：对比 prompt（ActAdd）、SAE 单义特征（精准可解释）、difference-of-means/CAA（强基线、稳健）。
- **强度有窗口**：太小没效果、太大崩坏（范数爆炸/困惑度飙升）；必须做剂量-反应扫描。
- **off-target 副作用**：方向越纯副作用越低；on-target vs off-target 又是一条帕累托权衡。
- **error-preserving clamp**：保留 SAE 重建残差，别让重建误差冒充副作用。
- **refusal direction**：单方向中介复杂行为——可加固也可越狱，interp 的双刃剑。

下一站：**模块 05 · 可解释性用于安全** —— 把读懂+控得动用于后门检测、model diffing 与 safety case。